In [ ]:
import pandas as pd
import networkx as nx
import itertools
import matplotlib.pyplot as plt
from networkx.algorithms import community

# 1. 加载数据 (增加 encoding='gbk' 解决报错)
try:
    df = pd.read_csv('data/milk_tea_brand_info.csv', encoding='gbk')
except UnicodeDecodeError:
    df = pd.read_csv('data/milk_tea_brand_info.csv', encoding='utf-8-sig')

df.shape

In [ ]:
print(df.nunique())

In [ ]:
df.isnull().sum()

In [ ]:
# 2. 构建网络：基于“商圈(business_zone)”的竞争关系
# 逻辑：如果两个品牌出现在同一个商圈，则建立一条边，权重为共同出现的次数
G = nx.Graph()
zones = df.groupby('business_zone')['brand'].unique()

for zone, brands in zones.items():
    if len(brands) > 1:
        # 两两品牌建立连接
        for u, v in itertools.combinations(brands, 2):
            if G.has_edge(u, v):
                G[u][v]['weight'] += 1
            else:
                G.add_edge(u, v, weight=1)

# 确保所有品牌都在节点中
all_brands = df['brand'].unique()
G.add_nodes_from(all_brands)

# 3. 计算社交网络核心指标
# (1) 加权度 (Weighted Degree): 反映竞争的激烈程度和品牌活跃度
degree = dict(G.degree(weight='weight'))

# (2) 集聚系数 (Clustering Coefficient): 反映品牌的“扎堆”程度
clustering = nx.clustering(G, weight='weight')

# (3) Katz 中心性: 衡量品牌的战略地位（不仅看对手多寡，还看对手的强度）
phi = max(nx.adjacency_spectrum(G).real) if len(G.edges()) > 0 else 1
alpha = 0.1 / phi if phi > 0 else 0.01
katz = nx.katz_centrality(G, alpha=alpha, weight='weight')

# (4) 社区发现与模块度 (Community & Modularity): 查看品牌是否形成了特定集群
comm_gen = community.greedy_modularity_communities(G, weight='weight')
partition = {node: i for i, comm in enumerate(comm_gen) for node in comm}
modularity_score = community.modularity(G, comm_gen, weight='weight')

# 4. 汇总指标报表
metrics_df = pd.DataFrame({
    '品牌': all_brands,
    '活跃度(加权度)': [degree.get(b, 0) for b in all_brands],
    '集聚系数': [round(clustering.get(b, 0), 3) for b in all_brands],
    'Katz中心性': [round(katz.get(b, 0), 3) for b in all_brands],
    '所属集群': [partition.get(b, -1) for b in all_brands]
}).sort_values('活跃度(加权度)', ascending=False)

print(f"整体网络模块度: {modularity_score:.4f}")
print(metrics_df)
metrics_df.to_csv('奶茶品牌社交网络分析结果.csv', index=False, encoding='gbk')

# 5. 可视化
plt.figure(figsize=(12, 8))
plt.rcParams['font.sans-serif'] = ['SimHei'] # 设置中文字体
plt.rcParams['axes.unicode_minus'] = False

pos = nx.spring_layout(G, k=0.6, seed=42)
# 节点大小由加权度决定，颜色由集群决定
nx.draw_networkx_nodes(G, pos, node_size=[v*30 + 300 for v in degree.values()], 
                       node_color=list(partition.values()), cmap=plt.cm.Set3)
# 边粗细由共现权重决定
weights = [G[u][v]['weight'] for u, v in G.edges()]
nx.draw_networkx_edges(G, pos, width=[w*1.2 for w in weights], alpha=0.4)
nx.draw_networkx_labels(G, pos, font_size=9)

plt.title('奶茶品牌商圈竞争/集群网络分析', fontsize=15)
plt.axis('off')
plt.show()

(1) 为什么模块度是 0？（竞争白热化的信号）
现象解释：模块度为 0 且所有品牌都在“社区 0”，说明这 10 个品牌在上海黄浦、静安两区的选址逻辑高度趋同。它们不是在互补，而是在核心商圈（如南京东路、静安寺、打浦桥）进行“短兵相接”的全面竞争。
分析视角：在这种情况下，传统的“社区划分”失效，我们应该转向
“核心-边缘”结构分析或“基于购物中心的颗粒度分析”
。
(2) 活跃度（度）与 Katz 中心性的含义
霸王茶姬 (95) 和 喜茶 (93)：这两个品牌拥有最高的加权度。在社交网络中，它们是“枢纽（Hubs）”。从竞争角度看，它们是
“全域掠夺者”
，几乎出现在所有热门商圈，是任何其他品牌都绕不开的对手。
Katz 中心性：由于大家的分值非常接近（0.30-0.32），说明市场结构非常稳定，不存在明显的孤立者。每个品牌都紧紧抱团在核心地带。
(3) 识别“影子竞争者”（Jaccard 相似度视角）
除了网络指标，建议引入 Jaccard 相似度。如果品牌 A 和品牌 B 的位置相似度极高（如霸王茶姬与喜茶），则存在明显的
“跟随策略”或“集群效应”
。

In [ ]:
# --- 核心步骤：为计算路径指标定义“距离” ---
# 权重(weight)越大代表竞争越直接，路径距离(distance)应该越短
for u, v, d in G.edges(data=True):
    d['distance'] = 1.0 / d['weight']

# 计算多维度社交网络指标
# (1) 加权度 (Strength/Weighted Degree)
degree_w = dict(G.degree(weight='weight'))

# (2) 介数中心性 (Betweenness): 衡量品牌作为“中转桥梁”的能力
# 使用 distance 权重计算最短路径
betweenness = nx.betweenness_centrality(G, weight='distance', normalized=True)

# (3) 接近中心性 (Closeness): 衡量品牌到达其他所有品牌的难易程度
closeness = nx.closeness_centrality(G, distance='distance')

# (4) 特征向量中心性 (Eigenvector): 衡量品牌是否连接到了其他“强力”品牌
# 直接使用 weight，因为权重越大影响力传导越强
eigenvector = nx.eigenvector_centrality(G, weight='weight', max_iter=1000)

# (5) 集聚系数 (Clustering)
clustering = nx.clustering(G, weight='weight')

# (6) Katz 中心性
phi = max(nx.adjacency_spectrum(G).real) if len(G.edges()) > 0 else 1
alpha = 0.1 / phi if phi > 0 else 0.01
katz = nx.katz_centrality(G, alpha=alpha, weight='weight')

# 汇总报表
metrics_df = pd.DataFrame({
    '品牌': all_brands,
    '加权度': [degree_w.get(b, 0) for b in all_brands],
    '介数中心性': [round(betweenness.get(b, 0), 4) for b in all_brands],
    '接近中心性': [round(closeness.get(b, 0), 4) for b in all_brands],
    '特征向量中心性': [round(eigenvector.get(b, 0), 4) for b in all_brands],
    '集聚系数': [round(clustering.get(b, 0), 3) for b in all_brands],
    'Katz中心性': [round(katz.get(b, 0), 3) for b in all_brands]
}).sort_values('加权度', ascending=False)

print(metrics_df)

针对商场内门店

In [ ]:
# 排除没有明确商场的门店数据
mall_df = df[df['mall_name'] != '无']

def analyze_network(data, group_col):
    G = nx.Graph()
    groups = data.groupby(group_col)['brand'].unique()
    
    for _, brands in groups.items():
        if len(brands) > 1:
            for u, v in itertools.combinations(brands, 2):
                if G.has_edge(u, v):
                    G[u][v]['weight'] += 1
                else:
                    G.add_edge(u, v, weight=1)
    
    # 计算指标
    degree = dict(G.degree(weight='weight'))
    clustering = nx.clustering(G, weight='weight')
    
    # 使用 Louvain 算法进行更细致的社区发现 (如果 networkx 版本支持)
    try:
        comms = community.louvain_communities(G, weight='weight', seed=42)
    except:
        comms = community.greedy_modularity_communities(G, weight='weight')
        
    partition = {node: i for i, c in enumerate(comms) for node in c}
    modularity = community.modularity(G, comms, weight='weight')
    
    return G, degree, clustering, partition, modularity

# 2. 执行 Mall 层级的分析
G_mall, deg, clust, part, mod = analyze_network(mall_df, 'mall_name')

# 3. 计算 Jaccard 相似度 (查看谁在跟谁走)
brand_locations = df.groupby('brand')['business_zone'].apply(set).to_dict()
jaccard_results = []
brands = list(brand_locations.keys())
for b1, b2 in itertools.combinations(brands, 2):
    s1, s2 = brand_locations[b1], brand_locations[b2]
    sim = len(s1 & s2) / len(s1 | s2)
    jaccard_results.append({'品牌1': b1, '品牌2': b2, '相似度': sim})

jaccard_df = pd.DataFrame(jaccard_results).sort_values('相似度', ascending=False)

# 4. 打印报告
print(f"购物中心网络模块度: {mod:.4f}")
print("\n--- 品牌相似度 Top 5 (跟随行为分析) ---")
print(jaccard_df.head(5))

A. “顶流双雄”与跟随策略（相似度 0.75）
现象：喜茶与霸王茶姬的相似度高达 0.75，且活跃度（加权度）领跑全场。
解读：这在社交网络中称为“结构对等性”。霸王茶姬在上海的扩张表现出极强的
“贴身肉搏”
倾向，几乎是“喜茶开在哪，我就开在哪”。这种跟随行为旨在截流同购买力的精准客群。
B. 两大阵营的界限（社区划分）
集群 0（购物中心派）：霸王茶姬、喜茶、爷爷不泡茶、茉莉奶白、乐乐茶。
这个阵营的品牌高度依赖 Mall 流量，选址偏向“品牌溢价”。
集群 1（高频刚需派）：一点点、茶百道、沪上阿姨、蜜雪冰城。
这个阵营的相似度也高，但它们更多出现在地铁站、街边、办公区底层。它们之间的竞争属于“存量竞争”，即谁的店多、谁出餐快。
C. 集聚系数（Clustering Coefficient）
所有品牌集聚系数都在 0.4-0.6 之间，这在社交网络中属于极高水平。
解读：这意味着“我对手的对手，也是我的对手”。上海的奶茶市场不是“一山不容二虎”，而是“一山必有群虎”。这种高度集聚有助于形成“奶茶聚集效应”，像餐饮层一样，通过集群吸引更多消费者。

In [ ]:
import seaborn as sns

# 1. 构建竞争强度矩阵 (以共同出现的商圈数量为指标)
brands = df['brand'].unique()
matrix = pd.DataFrame(0, index=brands, columns=brands)

for zone, group in df.groupby('business_zone'):
    zone_brands = group['brand'].unique()
    for b1 in zone_brands:
        for b2 in zone_brands:
            matrix.loc[b1, b2] += 1

# 2. 绘制热力图
plt.figure(figsize=(10, 8))
plt.rcParams['font.sans-serif'] = ['SimHei']
sns.heatmap(matrix, annot=True, cmap='YlGnBu', fmt='d')
plt.title('上海各区奶茶品牌“商圈共现”竞争热力图')
plt.xlabel('品牌')
plt.ylabel('品牌')
plt.savefig('competition_heatmap.png')

# 3. 绘制 Mall 网络社交图 (优化布局)
# (假设 G_mall 和 partition 已经按之前的逻辑计算好)
# 这里提供美化版绘图逻辑
plt.figure(figsize=(12, 10))
pos = nx.kamada_kawai_layout(G_mall) # 使用物理力导向算法，展示集群效果更好

# 绘制节点
colors = [partition.get(node, 0) for node in G_mall.nodes()]
nx.draw_networkx_nodes(G_mall, pos, node_size=1500, node_color=colors, cmap=plt.cm.Pastel1, edgecolors='gray')

# 绘制边 (权重越细竞争越弱，越粗竞争越强)
edges = G_mall.edges(data=True)
nx.draw_networkx_edges(G_mall, pos, width=[d['weight']*0.8 for u, v, d in edges], alpha=0.3)

# 标注标签
nx.draw_networkx_labels(G_mall, pos, font_family='SimHei', font_size=12)

plt.title('奶茶品牌社交网络分析：集群与竞争态势', fontsize=15)
plt.axis('off')

经纬度利用

In [ ]:
import numpy as np

# 剔除坐标缺失的行 (您的数据中约有7条缺失)
geo_df = df.dropna(subset=['latitude', 'longitude']).copy()

# 2. 定义球面距离计算函数 (Haversine)
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0  # 地球半径(km)
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2)**2
    return 2 * R * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

# 3. 计算空间竞争指标
# 设置竞争半径（例如 500米 = 0.5km）
RADIUS = 0.5 
comp_indices = []

for i, row in geo_df.iterrows():
    # 筛选其他品牌的门店
    mask = (geo_df['brand'] != row['brand'])
    others = geo_df[mask]
    
    # 计算当前店到其他所有店的距离
    dists = haversine(row['latitude'], row['longitude'], 
                      others['latitude'].values, others['longitude'].values)
    
    # 统计距离在 500m 以内的竞品店数量
    count = np.sum(dists <= RADIUS)
    comp_indices.append(count)

geo_df['spatial_competition'] = comp_indices

# 4. 聚合品牌层面指标
brand_spatial_metrics = geo_df.groupby('brand')['spatial_competition'].agg(['mean', 'max', 'sum']).reset_index()
brand_spatial_metrics.columns = ['品牌', '平均竞争密度', '最大竞争压力', '总竞争接触量']

print(brand_spatial_metrics.sort_values(by='平均竞争密度', ascending=False))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 8))
plt.rcParams['font.sans-serif'] = ['SimHei']

# 设置一个极小的偏移量（约 10-20米）
jitter = 0.0001 

for brand in geo_df['brand'].unique():
    data = geo_df[geo_df['brand'] == brand]
    # 在经纬度上加上随机扰动
    plt.scatter(data['longitude'] + np.random.uniform(-jitter, jitter, len(data)), 
                data['latitude'] + np.random.uniform(-jitter, jitter, len(data)), 
                label=brand, alpha=0.7, s=50)

plt.title('含扰动处理的奶茶门店分布图（防止点重叠）')
plt.legend(bbox_to_anchor=(1.05, 1))
plt.show()

结构空洞试分析

In [ ]:
# 1. 加载数据
try:
    df = pd.read_csv('data/milk_tea_brand_info.csv', encoding='utf-8')
except:
    df = pd.read_csv('data/milk_tea_brand_info.csv', encoding='gbk')

# 2. 构建网络的函数
def build_brand_network(dataframe):
    G = nx.Graph()
    # 按商圈分组，寻找品牌共现
    zones = dataframe.groupby('business_zone')['brand'].unique()
    for brands in zones:
        if len(brands) > 1:
            for u, v in itertools.combinations(brands, 2):
                if G.has_edge(u, v):
                    G[u][v]['weight'] += 1
                else:
                    G.add_edge(u, v, weight=1)
    # 确保所有品牌都作为节点存在
    all_brands = df['brand'].unique()
    G.add_nodes_from(all_brands)
    return G

# 3. 拆分数据集并构建网络
df_mall = df[df['in_mall'] == 1]
df_street = df[df['in_mall'] == 0]

G_mall = build_brand_network(df_mall)
G_street = build_brand_network(df_street)

# 4. 计算结构空洞指标函数
def compute_structural_holes(G):
    # 有效规模 (Effective Size)
    eff = nx.effective_size(G, weight='weight')
    # 限制性 (Constraint)
    cons = {}
    for n in G.nodes():
        if G.degree(n) > 0:
            # 这里的[n]表示只计算当前节点的限制性
            cons[n] = nx.constraint(G, [n], weight='weight')[n]
        else:
            cons[n] = 1.0 # 孤立节点限制性最大
    return eff, cons

# 5. 执行计算并整理结果
eff_m, cons_m = compute_structural_holes(G_mall)
eff_s, cons_s = compute_structural_holes(G_street)

results = pd.DataFrame({
    '品牌': list(G_mall.nodes()),
    'Mall_有效规模': [round(eff_m[n], 3) for n in G_mall.nodes()],
    'Mall_限制性': [round(cons_m[n], 3) for n in G_mall.nodes()],
    'Street_有效规模': [round(eff_s[n], 3) for n in G_mall.nodes()],
    'Street_限制性': [round(cons_s[n], 3) for n in G_mall.nodes()]
}).sort_values('Mall_有效规模', ascending=False)

print("结构空洞分析：Mall vs 街边店")
print(results)

考虑区分布

In [ ]:
# 1. 定义分析函数
def get_district_metrics(district_name, df):
    # 筛选区数据
    dist_df = df[df['district'] == district_name]
    G = nx.Graph()
    
    # 构建商圈竞争网络
    zones = dist_df.groupby('business_zone')['brand'].unique()
    for brands in zones:
        if len(brands) > 1:
            for u, v in itertools.combinations(brands, 2):
                if G.has_edge(u, v):
                    G[u][v]['weight'] += 1
                else:
                    G.add_edge(u, v, weight=1)
    
    # 基础指标
    all_brands = df['brand'].unique()
    G.add_nodes_from(all_brands)
    
    density = nx.density(G)
    avg_deg = sum(dict(G.degree(weight='weight')).values()) / len(all_brands)
    avg_clust = sum(nx.clustering(G, weight='weight').values()) / len(all_brands)
    
    # 模块度 (使用贪婪算法)
    comms = community.greedy_modularity_communities(G, weight='weight')
    mod = community.modularity(G, comms, weight='weight')
    
    return mod, density, avg_deg, avg_clust

# 2. 执行对比
hp_metrics = get_district_metrics('黄浦区', df)
ja_metrics = get_district_metrics('静安区', df)

# 3. 结果汇总
comparison_df = pd.DataFrame({
    '区域': ['黄浦区', '静安区'],
    '模块度': [hp_metrics[0], ja_metrics[0]],
    '网络密度': [hp_metrics[1], ja_metrics[1]],
    '平均竞争强度(加权度)': [hp_metrics[2], ja_metrics[2]],
    '平均集聚系数': [hp_metrics[3], ja_metrics[3]]
})

print(comparison_df)

In [ ]:
def plot_district_network(district_name, filename):
    # 筛选特定区数据
    dist_df = df[df['district'] == district_name]
    G = nx.Graph()
    
    # 构建边：基于商圈共现
    zones = dist_df.groupby('business_zone')['brand'].unique()
    for brands in zones:
        if len(brands) > 1:
            for u, v in itertools.combinations(brands, 2):
                if G.has_edge(u, v):
                    G[u][v]['weight'] += 1
                else:
                    G.add_edge(u, v, weight=1)
    
    # 确保所有品牌作为节点出现
    G.add_nodes_from(all_brands)
    
    # 社区发现用于节点着色
    comms = community.greedy_modularity_communities(G, weight='weight')
    partition = {node: i for i, c in enumerate(comms) for node in c}
    
    # 可视化设置
    plt.figure(figsize=(10, 8))
    plt.rcParams['font.sans-serif'] = ['SimHei']
    plt.rcParams['axes.unicode_minus'] = False
    
    pos = nx.circular_layout(G) # 环形布局方便对比连接密度
    
    # 节点大小由加权度决定
    degrees = dict(G.degree(weight='weight'))
    node_sizes = [degrees.get(b, 0) * 50 + 500 for b in G.nodes()]
    
    # 绘制
    nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color=list(partition.values()), 
                           cmap=plt.cm.Set3, edgecolors='gray')
    
    if G.number_of_edges() > 0:
        weights = [G[u][v]['weight'] for u, v in G.edges()]
        max_w = max(weights)
        # 边粗细随竞争权重变化
        nx.draw_networkx_edges(G, pos, width=[(w / max_w) * 5 for w in weights], alpha=0.3)
        
    nx.draw_networkx_labels(G, pos, font_size=10)
    
    plt.title(f'奶茶品牌竞争网络分析 - {district_name}', fontsize=15)
    plt.axis('off')
    plt.savefig(filename)

# 生成图片
plot_district_network('黄浦区', 'huangpu_network.png')
plot_district_network('静安区', 'jingan_network.png')